# Resident Risk of Regression Prediction
Predicting which residents are at risk of regression (deteriorating outcomes) during or after their shelter stay.

## 1. Problem Framing
Resident regression refers to a measurable decline in wellbeing indicators (mental health, employment status, housing stability, substance use relapse) after a period of improvement. Early identification allows proactive intervention before full crisis reoccurs.

**Target variable:** `regression_score` — continuous index (0–100; higher = greater regression risk) derived from composite wellbeing indicators. We also train a binary classifier using a threshold of 60.

**Approach:** Regression model to predict the continuous score, plus binary classification for actionable risk tiers, plus a causal analysis using SHAP to identify the dominant drivers.

**Success metrics:**
- Regression: R² ≥ 0.65, RMSE ≤ 12 points.
- Classification (high risk ≥ 60): ROC-AUC ≥ 0.80.

**Stakeholders:** Case managers, program coordinators, clinical support staff.

## 2. Data Acquisition & Preparation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 750

df = pd.DataFrame({
    'resident_id': range(1, n + 1),
    'age': np.random.randint(18, 70, size=n),
    'length_of_stay_days': np.random.randint(7, 730, size=n),
    'mental_health_baseline': np.random.randint(1, 11, size=n),  # 1=poor,10=excellent
    'mental_health_current': np.random.randint(1, 11, size=n),
    'substance_use_baseline': np.random.randint(0, 11, size=n),  # 0=none,10=severe
    'substance_use_current': np.random.randint(0, 11, size=n),
    'employment_change': np.random.choice([-1, 0, 1], size=n, p=[0.25, 0.50, 0.25]),  # -1=lost, 0=same, 1=gained
    'missed_appointments_last30': np.random.randint(0, 8, size=n),
    'social_support_score': np.random.randint(0, 11, size=n),
    'housing_stability_score': np.random.randint(0, 11, size=n),
    'crisis_events_last90': np.random.randint(0, 6, size=n),
    'program_attendance_rate': np.random.uniform(0, 1, size=n),
    'prior_regression_episodes': np.random.randint(0, 5, size=n),
    'peer_conflict_flag': np.random.randint(0, 2, size=n),
})

# Derived feature: change scores
df['mental_health_delta'] = df['mental_health_current'] - df['mental_health_baseline']
df['substance_use_delta'] = df['substance_use_current'] - df['substance_use_baseline']

# Simulate continuous regression score (0–100)
raw_score = (
    40
    - 3.0 * df['mental_health_delta']          # improvement reduces risk
    + 4.0 * df['substance_use_delta']           # worsening increases risk
    - 2.0 * df['employment_change']
    + 5.0 * df['missed_appointments_last30']
    - 1.5 * df['social_support_score']
    - 1.5 * df['housing_stability_score']
    + 8.0 * df['crisis_events_last90']
    - 10.0 * df['program_attendance_rate']
    + 6.0 * df['prior_regression_episodes']
    + 5.0 * df['peer_conflict_flag']
    + np.random.normal(0, 5, size=n)
).clip(0, 100)

df['regression_score'] = raw_score
df['high_risk'] = (raw_score >= 60).astype(int)

print(df.shape)
print(f"High risk rate: {df['high_risk'].mean():.2%}")
print(f"Regression score — mean: {df['regression_score'].mean():.1f}, std: {df['regression_score'].std():.1f}")
df.head()

In [ ]:
feature_cols = [
    'age', 'length_of_stay_days', 'mental_health_baseline', 'mental_health_current',
    'mental_health_delta', 'substance_use_baseline', 'substance_use_current',
    'substance_use_delta', 'employment_change', 'missed_appointments_last30',
    'social_support_score', 'housing_stability_score', 'crisis_events_last90',
    'program_attendance_rate', 'prior_regression_episodes', 'peer_conflict_flag'
]

X = df[feature_cols]
y_reg = df['regression_score']
y_cls = df['high_risk']

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, stratify=y_cls, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Exploration

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Regression score distribution
axes[0, 0].hist(df['regression_score'], bins=30, color='tomato', edgecolor='white')
axes[0, 0].axvline(60, color='black', linestyle='--', label='High-risk threshold')
axes[0, 0].set_title('Regression Score Distribution')
axes[0, 0].set_xlabel('Regression Score')
axes[0, 0].legend()

# Score vs crisis events
axes[0, 1].scatter(df['crisis_events_last90'], df['regression_score'],
                   alpha=0.3, s=10, c=df['high_risk'], cmap='coolwarm')
axes[0, 1].set_title('Crisis Events vs Regression Score')
axes[0, 1].set_xlabel('Crisis Events (last 90 days)')
axes[0, 1].set_ylabel('Regression Score')

# Score vs program attendance
axes[0, 2].scatter(df['program_attendance_rate'], df['regression_score'],
                   alpha=0.3, s=10, c=df['high_risk'], cmap='coolwarm')
axes[0, 2].set_title('Program Attendance vs Regression Score')
axes[0, 2].set_xlabel('Attendance Rate')
axes[0, 2].set_ylabel('Regression Score')

# Mental health delta by risk
for risk, label, color in [(0, 'Low risk', 'steelblue'), (1, 'High risk', 'tomato')]:
    sub = df[df['high_risk'] == risk]
    axes[1, 0].hist(sub['mental_health_delta'], bins=15, alpha=0.6, label=label, color=color)
axes[1, 0].set_title('Mental Health Delta by Risk')
axes[1, 0].legend()

# Substance use delta by risk
for risk, label, color in [(0, 'Low risk', 'steelblue'), (1, 'High risk', 'tomato')]:
    sub = df[df['high_risk'] == risk]
    axes[1, 1].hist(sub['substance_use_delta'], bins=15, alpha=0.6, label=label, color=color)
axes[1, 1].set_title('Substance Use Delta by Risk')
axes[1, 1].legend()

# Prior regression episodes
for risk, label, color in [(0, 'Low risk', 'steelblue'), (1, 'High risk', 'tomato')]:
    sub = df[df['high_risk'] == risk]
    axes[1, 2].hist(sub['prior_regression_episodes'], bins=6, alpha=0.6, label=label, color=color)
axes[1, 2].set_title('Prior Regression Episodes by Risk')
axes[1, 2].legend()

plt.suptitle('Resident Regression Risk — EDA', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Modeling
### 4a. Predictive Model — Gradient Boosting Regressor (continuous score) + Random Forest Classifier (high-risk binary)
### 4b. Causal / Explanatory Model — SHAP (TreeExplainer on regressor)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier

# --- 4a: Regression model for continuous score ---
gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gbr.fit(X_train, y_reg_train)
y_reg_pred = gbr.predict(X_test)

# --- Classification model for high-risk flag ---
rf_cls = RandomForestClassifier(
    n_estimators=200, max_depth=6, random_state=42, class_weight='balanced'
)
rf_cls.fit(X_train, y_cls_train)
y_cls_proba = rf_cls.predict_proba(X_test)[:, 1]
y_cls_pred = rf_cls.predict(X_test)

print("Regressor and classifier training complete.")

In [ ]:
# --- 4b: Causal / Explanatory model using SHAP on the regressor ---
try:
    import shap

    explainer = shap.TreeExplainer(gbr)
    shap_values = explainer.shap_values(X_test)

    # Global summary
    shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
    plt.title('SHAP Summary — Resident Regression Score (higher = more risk)')
    plt.tight_layout()
    plt.show()

    # Bar plot of mean absolute SHAP values
    mean_abs_shap = pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=feature_cols
    ).sort_values(ascending=False)

    plt.figure(figsize=(9, 4))
    mean_abs_shap.head(10).plot(kind='bar', color='tomato')
    plt.title('Mean |SHAP| — Top 10 Risk Drivers')
    plt.ylabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.show()

    print("Top 5 risk drivers by mean |SHAP|:")
    print(mean_abs_shap.head(5).round(4))
except ImportError:
    print("shap not installed — skipping. Install with: pip install shap")

## 5. Evaluation

In [ ]:
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    roc_auc_score, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay
)

# Regression evaluation
r2 = r2_score(y_reg_test, y_reg_pred)
rmse = mean_squared_error(y_reg_test, y_reg_pred, squared=False)
mae = mean_absolute_error(y_reg_test, y_reg_pred)
print(f"Regression — R²: {r2:.4f}, RMSE: {rmse:.2f} pts, MAE: {mae:.2f} pts")
print()

# Classification evaluation
print(f"Classification ROC-AUC: {roc_auc_score(y_cls_test, y_cls_proba):.4f}")
print(classification_report(y_cls_test, y_cls_pred, target_names=['Low Risk', 'High Risk']))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Predicted vs actual (regression)
axes[0].scatter(y_reg_test, y_reg_pred, alpha=0.3, s=8, color='tomato')
mn, mx = 0, 100
axes[0].plot([mn, mx], [mn, mx], 'k--')
axes[0].set_xlabel('Actual Regression Score')
axes[0].set_ylabel('Predicted Regression Score')
axes[0].set_title(f'Predicted vs Actual (R²={r2:.3f})')

RocCurveDisplay.from_predictions(y_cls_test, y_cls_proba, ax=axes[1], name='RF Classifier')
axes[1].set_title('ROC Curve — High Risk Classification')

ConfusionMatrixDisplay.from_predictions(
    y_cls_test, y_cls_pred, display_labels=['Low Risk', 'High Risk'], ax=axes[2]
)
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

## 6. Feature Selection

In [ ]:
importances_reg = pd.Series(gbr.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances_cls = pd.Series(rf_cls.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
importances_reg.head(10).plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Regressor Feature Importances (top 10)')
axes[0].set_ylabel('Importance')

importances_cls.head(10).plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Classifier Feature Importances (top 10)')
axes[1].set_ylabel('Importance')

plt.tight_layout()
plt.show()

# Union of top features from both models
top_reg = set(importances_reg.head(8).index)
top_cls = set(importances_cls.head(8).index)
selected_features = list(top_reg.union(top_cls))
print(f"Selected features ({len(selected_features)} total):", sorted(selected_features))

## 7. Deployment

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

# Retrain both final models on selected features
gbr_final = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, random_state=42
)
gbr_final.fit(X_train_sel, y_reg_train)

rf_final = RandomForestClassifier(
    n_estimators=200, max_depth=6, random_state=42, class_weight='balanced'
)
rf_final.fit(X_train_sel, y_cls_train)

with open('models/resident_risk_regression_model.pkl', 'wb') as f:
    pickle.dump({
        'regressor': gbr_final,
        'classifier': rf_final,
        'features': selected_features,
        'scaler': scaler
    }, f)

print("Models saved to models/resident_risk_regression_model.pkl")

# Generate risk dashboard row for a sample of residents
sample = X_test_sel.iloc[:10].copy()
sample['regression_score_predicted'] = gbr_final.predict(sample).clip(0, 100).round(1)
sample['high_risk_probability'] = rf_final.predict_proba(sample)[:, 1].round(3)
sample['risk_tier'] = pd.cut(
    sample['high_risk_probability'],
    bins=[0, 0.30, 0.60, 1.01],
    labels=['Low', 'Medium', 'High']
)

print("\nRisk dashboard — sample residents:")
print(sample[['regression_score_predicted', 'high_risk_probability', 'risk_tier']].to_string())